# Load a raw traceability dataset into our schema

Converts a raw dataset (iTrust, CM-1, eTour, LibEST) into `Artifact` and `TraceabilityLink` records.
Raw files are only read, never changed. The result goes to `data/processed/<dataset>/`:

| File | Content |
|---|---|
| `artifacts.json` | every requirement and target (code or design) as an `Artifact` |
| `traces.json` | one `TraceabilityLink` per ground-truth link |
| `gold.json` | `[source_id, target_id, is_true_link]` for every requirement × target pair, for `agents.traceability_agent.evaluate --gold` |

**Mapping decisions**

- **iTrust:** "Use Cases" become requirements; "Java Code" and "JSP Code" become code. Links run use case → code.
- **CM-1:** High Level Requirements (SRS…) become requirements; Low Level Requirements (DPUSDS…) become design. Links are flipped to run requirement → design.
- **Gold set:** every pair not marked true is a non-link, so `gold.json` holds the full requirement × target cross product. Without the negatives, false positives could never be counted and precision would be meaningless.
- **Text:** prose is whitespace-collapsed; code keeps its line structure (only line endings, trailing spaces and blank-line runs are normalised).

**How to use:** choose the dataset in the settings cell, run the cells top to bottom. The last cell is the only one that writes files.

In [ ]:
from pathlib import Path
import sys

# the notebook may be opened from scripts/ or from the project folder: find the project root either way
here = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in [here, *here.parents] if (p / "agents").is_dir() and (p / "shared").is_dir())
sys.path.insert(0, str(PROJECT_ROOT))
print("project root:", PROJECT_ROOT)

In [ ]:
import csv
import json
import unicodedata
from itertools import product
from typing import Callable, Optional, Sequence

from shared.schemas.traceability import Artifact, TraceabilityLink

DEFAULT_RAW = Path.home() / "Downloads" / "traceability_datasets"
CM1_LAYERS = {"High Level Requirements": "requirement", "Low Level Requirements": "design"}
ITRUST_LAYERS = {"Use Cases": "requirement", "Java Code": "code", "JSP Code": "code"}

## Readers and helpers
One reader per dataset, plus the functions that clean the text and build the links and the gold set.

In [ ]:
def normalise_prose(text: str) -> str:
    """NFC-normalise and collapse all whitespace runs to single spaces."""
    return " ".join(unicodedata.normalize("NFC", text).split())


def normalise_code(text: str) -> str:
    """NFC-normalise, unify line endings, strip trailing spaces, cap blank-line runs at one."""
    lines = [ln.rstrip() for ln in unicodedata.normalize("NFC", text).replace("\r\n", "\n").replace("\r", "\n").split("\n")]
    out: list[str] = []
    for ln in lines:
        if ln == "" and out and out[-1] == "":
            continue
        out.append(ln)
    return "\n".join(out).strip()


def make_artifact(
    seen: dict[str, Artifact], art_id: str, layer: str, content: str, layers: dict[str, str], dataset: str
) -> None:
    """Normalise one raw row into ``seen`` (first occurrence of an id wins)."""
    if layer not in layers:
        raise ValueError(f"Unknown layer {layer!r} for {art_id}")
    type_ = layers[layer]
    text = normalise_prose(content) if type_ != "code" else normalise_code(content)
    if art_id in seen:
        if seen[art_id].text != text:
            print(f"warning: duplicate id {art_id!r} with different text; keeping first")
        return
    metadata = {"dataset": dataset, "layer": layer}
    if layer == "JSP Code":
        metadata["language"] = "jsp"
    elif layer == "Java Code":
        metadata["language"] = "java"
    seen[art_id] = Artifact(id=art_id, type=type_, text=text, metadata=metadata)


def load_cm1(raw: Path) -> tuple[list[Artifact], set[tuple[str, str]]]:
    """Read CM-1 (artifacts.csv + traces.csv) as artifacts and (requirement, design) true pairs."""
    seen: dict[str, Artifact] = {}
    with open(raw / "artifacts.csv", encoding="utf-8-sig", newline="") as fh:
        for row in csv.DictReader(fh):
            make_artifact(seen, row["id"].strip(), row["layer"], row["content"], CM1_LAYERS, "CM-1")
    pairs: set[tuple[str, str]] = set()
    with open(raw / "traces.csv", encoding="utf-8-sig", newline="") as fh:
        for row in csv.DictReader(fh):
            if row["label"].strip() != "1":
                continue
            low, high = row["s_id"].strip(), row["t_id"].strip()
            if low not in seen or high not in seen:
                raise ValueError(f"Trace references unknown artifact: {low} -> {high}")
            if seen[high].type != "requirement" or seen[low].type != "design":
                raise ValueError(f"Trace has unexpected layers: {low} -> {high}")
            pairs.add((high, low))  # flipped to requirement -> design
    return list(seen.values()), pairs


def load_itrust(raw: Path) -> tuple[list[Artifact], set[tuple[str, str]]]:
    """Read iTrust (Parquet) as artifacts and (use case, code) true pairs."""
    import pyarrow.parquet as pq  # only needed for the Parquet datasets

    def table(name: str) -> list[dict]:
        (path,) = sorted((raw / "iTrust" / name).glob("*.parquet"))
        return pq.read_table(path).to_pylist()

    seen: dict[str, Artifact] = {}
    for row in table("artifacts"):
        make_artifact(seen, row["id"], row["layer"], row["content"], ITRUST_LAYERS, "iTrust")
    pairs: set[tuple[str, str]] = set()
    for row in table("traces"):
        if row["label"] != 1:
            continue
        s, t = row["source"], row["target"]
        if s not in seen or t not in seen:
            raise ValueError(f"Trace references unknown artifact: {s} -> {t}")
        if seen[s].type != "requirement" or seen[t].type != "code":
            raise ValueError(f"Trace has unexpected layers: {s} -> {t}")
        pairs.add((s, t))
    return list(seen.values()), pairs


FTLR_DIR = "finegrained-traceability/datasets"


def load_ftlr(raw: Path, name: str, links_file: str, code_language: str) -> tuple[list[Artifact], set[tuple[str, str]]]:
    """Read a dataset from the FTLR repo (tobhey/finegrained-traceability).

    Layout: ``<name>/req/*.txt`` requirements, ``<name>/code/*`` code files, and a links
    file with lines ``REQ.txt: Target1 Target2 ...`` (whitespace-separated targets).
    Artifact ids are the file names, so the links file matches them exactly.
    """
    base = raw / FTLR_DIR / name
    seen: dict[str, Artifact] = {}
    for path in sorted((base / "req").iterdir()):
        text = path.read_text(encoding="utf-8-sig", errors="replace")
        seen[path.name] = Artifact(id=path.name, type="requirement", text=normalise_prose(text),
                                   metadata={"dataset": name, "layer": "requirement"})
    for path in sorted((base / "code").iterdir()):
        text = path.read_text(encoding="utf-8-sig", errors="replace")
        seen[path.name] = Artifact(id=path.name, type="code", text=normalise_code(text),
                                   metadata={"dataset": name, "layer": "code", "language": code_language})
    pairs: set[tuple[str, str]] = set()
    missing = 0
    for line in (base / links_file).read_text(encoding="utf-8-sig").splitlines():
        if ":" not in line:
            continue
        req, targets = line.split(":", 1)
        for target in targets.split():
            if req.strip() in seen and target in seen:
                pairs.add((req.strip(), target))
            else:
                missing += 1
    if missing:
        print(f"warning: {missing} link endpoints not found among the artifacts and were dropped")
    return list(seen.values()), pairs


def load_etour(raw: Path) -> tuple[list[Artifact], set[tuple[str, str]]]:
    """eTour: use cases -> Java classes (English version of the answer matrix)."""
    return load_ftlr(raw, "eTour", "etour_solution_links_english.txt", "java")


def load_libest(raw: Path) -> tuple[list[Artifact], set[tuple[str, str]]]:
    """LibEST: requirements (RFC text) -> C source and header files."""
    return load_ftlr(raw, "LibEST", "req_to_code_ground.txt", "c")


DATASETS: dict[str, Callable[[Path], tuple[list[Artifact], set[tuple[str, str]]]]] = {
    "cm1": load_cm1,
    "itrust": load_itrust,
    "etour": load_etour,
    "libest": load_libest,
}


def build_outputs(
    artifacts: Sequence[Artifact], true_pairs: set[tuple[str, str]]
) -> tuple[list[TraceabilityLink], list[list]]:
    """Build ground-truth links and the full labelled gold set (all requirement x target pairs)."""
    types = {a.id: a.type for a in artifacts}
    reqs = [a.id for a in artifacts if a.type == "requirement"]
    targets = [a.id for a in artifacts if a.type != "requirement"]
    links = [
        TraceabilityLink(
            source_id=s,
            target_id=t,
            link_type=f"requirement_to_{types[t]}",
            confidence=1.0,
            justification="Ground truth: expert-labelled link in the dataset.",
        )
        for s, t in sorted(true_pairs)
    ]
    gold = [[s, t, (s, t) in true_pairs] for s, t in product(reqs, targets)]
    return links, gold

## Settings
Choose the dataset (`cm1`, `itrust`, `etour` or `libest`) and where the raw files are.

In [ ]:
DATASET = "itrust"          # one of: cm1, itrust, etour, libest
RAW = DEFAULT_RAW           # folder with the raw dataset files
OUT = PROJECT_ROOT / "data" / "processed" / DATASET

## 1 · Read the raw dataset and build the links and the gold set
Nothing is written yet.

In [ ]:
artifacts, true_pairs = DATASETS[DATASET](RAW)
links, gold = build_outputs(artifacts, true_pairs)

n_req = sum(a.type == "requirement" for a in artifacts)
linked_reqs = len({s for s, _ in true_pairs})
print(f"{DATASET}: {len(artifacts)} artifacts ({n_req} requirement, {len(artifacts) - n_req} targets)")
print(f"true links: {len(links)} (requirements with >=1 link: {linked_reqs}/{n_req})")
print(f"gold pairs: {len(gold)} ({len(gold) - len(links)} negatives)")

## 2 · Write `data/processed/<dataset>/`
This cell replaces the three files of the chosen dataset.

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "artifacts.json").write_text(json.dumps([a.model_dump() for a in artifacts], indent=1, ensure_ascii=False), encoding="utf-8")
(OUT / "traces.json").write_text(json.dumps([l.model_dump() for l in links], indent=1, ensure_ascii=False), encoding="utf-8")
(OUT / "gold.json").write_text(json.dumps(gold), encoding="utf-8")
print("wrote", OUT)